# Практическая работа №4: Элементы корреляционного анализа. Проверка статистической гипотезы о равенстве коэффициента корреляции нулю

Выполнили студенты гр. 2384 Поглазов Никита Васильевич и Вовченко София Евгеньевна. Вариант 14.

## Цель работы

Освоение основных понятий, связанных с корреляционной зависимостью между случайными величинами, статистическими гипотезами и проверкой их "справедливости".

## Основные теоретические положения

**Корреляционная зависимость** характеризует наличие статистической связи между двумя случайными величинами $X$ и $Y$.

**Выборочный коэффициент корреляции** определяется формулой

$$
r_{в}=\frac{\sum\limits_{i=1}^{k_x}\sum\limits_{j=1}^{k_y} n_{ij}(x_i-\bar{x}_{в})(y_j-\bar{y}_{в})}{N\,\sigma_x\sigma_y}
$$

Здесь $n_{ij}$ — частота наблюдений в ячейке корреляционной таблицы, $x_i$ и $y_j$ — середины соответствующих интервалов, $N$ — объём выборки.

Для упрощения вычислений используют **условные варианты**, вводя величины

$$
u_i =
\frac{x_i-C_x}{h_x}, \qquad 
v_j = \frac{y_j-C_y}{h_y},
$$

где $C_x, C_y$ — условные начала отсчёта, а $h_x, h_y$ — длины интервалов группировки. В этом случае коэффициент корреляции можно записать в виде

$$
r_{в}=
\frac{m_{11}-m_{10}m_{01}}
{\sqrt{m_{20}-m_{10}^2}\sqrt{m_{02}-m_{01}^2}}
$$

**Доверительный интервал** для коэффициента корреляции строится с использованием преобразования Фишера:

$$
z=\frac{1}{2}\ln\frac{1+r_{в}}{1-r_{в}},
\qquad
z \pm u_{\gamma}\frac{1}{\sqrt{N-3}}
$$

После нахождения границ интервала выполняется обратное преобразование:

$$
r = \mathrm{th}(z)
$$

Для проверки статистической гипотезы

$$
H_0:\; r = 0, 
\qquad 
H_1:\; r \ne 0
$$

применяется статистика

$$
t_{набл}=\frac{r_{в}\sqrt{N-2}}{\sqrt{1-r_{в}^2}}
$$

Если выполняется условие $ |t_{набл}| > t_{кр} $, нулевая гипотеза отвергается.

## Постановка задачи

Из заданной генеральной совокупности сформировать выборку по второму признаку. Провести статистическую обработку второй выборки в объёме практических работ №1 и №2, с целью определения точечных статистических оценок параметров распределения исследуемого признака (математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса и коэффициента вариации). Для системы двух случайных величин $ X $ (первый признак) и $ Y $ (второй признак) сформировать двумерную выборку и найти статистическую оценку коэффициента корреляции, построить доверительный интервал для коэффициента корреляции и осуществить проверку статистической гипотезы о равенстве коэффициента корреляции нулю. Полученные результаты содержательно проинтерпретировать.


## Выполнение работы

- Провести статистическую обработку второй выборки в объёме практических работ №1 и №2, с целью определения точечных статистических оценок параметров распределения исследуемого признака (математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации). Оформить результаты в виде таблицы, сделать выводы.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import math
from IPython.display import display_markdown

In [2]:
df = pd.read_csv('../samples.csv')
X = df['SepalLengthCm'].to_numpy()
Y = df['SepalWidthCm'].to_numpy()

n = len(Y)
mean_Y = np.mean(Y)
var_Y = np.var(Y, ddof=1)
std_Y = np.std(Y, ddof=1)
skew_Y = stats.skew(Y, bias=False)
kurtosis_Y = stats.kurtosis(Y, bias=False)
mode_Y = stats.mode(Y, keepdims=True).mode[0]
median_Y = np.median(Y)
cv_Y = (std_Y / mean_Y) * 100

results = pd.DataFrame([{
    r'$\mu_y$': round(mean_Y, 4),
    r'$D_y$': round(var_Y, 4),
    r'$\sigma_y$': round(std_Y, 4),
    r'$A_s$': round(skew_Y, 4),
    r'$E_k$': round(kurtosis_Y, 4),
    r'$M_o$': round(mode_Y, 4),
    r'$M_e$': round(median_Y, 4),
    r'$V_y$': round(cv_Y, 4)
}])

display_markdown("### Точечные статистические оценки признака $Y$ (Ширина чашелистика)", raw=True)
display_markdown(results.to_markdown(index=False), raw=True)

### Точечные статистические оценки признака $Y$ (Ширина чашелистика)

|   $\mu_y$ |   $D_y$ |   $\sigma_y$ |   $A_s$ |   $E_k$ |   $M_o$ |   $M_e$ |   $V_y$ |
|----------:|--------:|-------------:|--------:|--------:|--------:|--------:|--------:|
|    3.0579 |  0.2047 |       0.4525 |  0.4385 |  0.3697 |       3 |       3 | 14.7964 |

Положительное значение асимметрии свидетельствует о том, что распределение исследуемого признака является правосторонне асимметричным. Положительное значение эксцесса говорит о том, что распределение исследуемого признака является более острым, чем нормальное распределение. Мода и медиана совпадают. 

- Построить двумерный интервальный вариационный ряд, оформить в виде таблицы.


In [3]:
def calculate_bins(data: np.ndarray) -> tuple[np.ndarray, float]:
    k = math.floor(1 + 3.322 * math.log10(len(data)))
    h = (np.max(data) - np.min(data)) / k
    edges = np.linspace(np.min(data), np.max(data), k + 1)
    return edges, h

x_edges, h_x = calculate_bins(X)
y_edges, h_y = calculate_bins(Y)

var_table, x_bin_edges, y_bin_edges = np.histogram2d(X, Y, bins=[x_edges, y_edges])

df_var = pd.DataFrame(var_table.astype(int), 
                      index=[f"[{x_bin_edges[i]:.2f}, {x_bin_edges[i+1]:.2f})" for i in range(len(x_bin_edges) - 1)],
                      columns=[f"[{y_bin_edges[i]:.2f}, {y_bin_edges[i+1]:.2f})" for i in range(len(y_bin_edges) - 1)])

display_markdown(df_var.to_markdown(), raw=True)

|              |   [2.00, 2.34) |   [2.34, 2.69) |   [2.69, 3.03) |   [3.03, 3.37) |   [3.37, 3.71) |   [3.71, 4.06) |   [4.06, 4.40) |
|:-------------|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|
| [4.30, 4.81) |              0 |              0 |              3 |              4 |              4 |              0 |              0 |
| [4.81, 5.33) |              1 |              2 |              3 |              4 |              7 |              1 |              1 |
| [5.33, 5.84) |              1 |              5 |              9 |              0 |              3 |              4 |              2 |
| [5.84, 6.36) |              3 |              2 |             11 |              3 |              1 |              0 |              0 |
| [6.36, 6.87) |              0 |              1 |             12 |              5 |              0 |              0 |              0 |
| [6.87, 7.39) |              0 |              0 |              3 |              4 |              1 |              0 |              0 |
| [7.39, 7.90) |              0 |              1 |              4 |              0 |              0 |              2 |              0 |

- По полученному двумерному интервальному вариационному ряду построить корреляционную таблицу, сделать выводы.


In [4]:
df_corr = df_var.copy()
df_corr[r'$\sum$'] = df_var.sum(axis=1)
df_corr.loc[r'$\sum$'] = df_var.sum(axis=0)
display_markdown(df_corr.to_markdown(), raw=True)

|              |   [2.00, 2.34) |   [2.34, 2.69) |   [2.69, 3.03) |   [3.03, 3.37) |   [3.37, 3.71) |   [3.71, 4.06) |   [4.06, 4.40) |   $\sum$ |
|:-------------|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------------:|---------:|
| [4.30, 4.81) |              0 |              0 |              3 |              4 |              4 |              0 |              0 |       11 |
| [4.81, 5.33) |              1 |              2 |              3 |              4 |              7 |              1 |              1 |       19 |
| [5.33, 5.84) |              1 |              5 |              9 |              0 |              3 |              4 |              2 |       24 |
| [5.84, 6.36) |              3 |              2 |             11 |              3 |              1 |              0 |              0 |       20 |
| [6.36, 6.87) |              0 |              1 |             12 |              5 |              0 |              0 |              0 |       18 |
| [6.87, 7.39) |              0 |              0 |              3 |              4 |              1 |              0 |              0 |        8 |
| [7.39, 7.90) |              0 |              1 |              4 |              0 |              0 |              2 |              0 |        7 |
| $\sum$       |              5 |             11 |             45 |             20 |             16 |              7 |              3 |      nan |

Построенная корреляционная таблица демонстрируют совместное распределение частот случайных величин $X$ и $Y$. Заметна большая концентрация наблюдений в определённых ячейках, что может указывать на наличие слабой корреляционной связи между этими признаками.

- Исходя из результатов корреляционной таблицы вычислить значение выборочного коэффициента корреляции двумя способами: с помощью стандартной формулы и с помощью условных вариант. Убедиться, что результаты совпадают. Сделать выводы.


In [11]:
# Способ 1
x_mid = (x_edges[:-1] + x_edges[1:]) / 2
y_mid = (y_edges[:-1] + y_edges[1:]) / 2

row_totals = var_table.sum(axis=1) # n_x
col_totals = var_table.sum(axis=0) # n_y
N = len(X)

x_bar_group = (x_mid * row_totals).sum() / N
y_bar_group = (y_mid * col_totals).sum() / N

sigma_x_group = math.sqrt((((x_mid - x_bar_group) ** 2) * row_totals).sum() / N)
sigma_y_group = math.sqrt((((y_mid - y_bar_group) ** 2) * col_totals).sum() / N)

numerator = 0.0
for i, xi in enumerate(x_mid):
    for j, yj in enumerate(y_mid):
        numerator += var_table[i, j] * (xi - x_bar_group) * (yj - y_bar_group)

r_standard = numerator / (N * sigma_x_group * sigma_y_group)

# Способ 2
C_x = x_mid[len(x_mid) // 2]
C_y = y_mid[len(y_mid) // 2]

u = (x_mid - C_x) / h_x
v = (y_mid - C_y) / h_y

m10 = (u * row_totals).sum() / N
m01 = (v * col_totals).sum() / N
m20 = ((u ** 2) * row_totals).sum() / N
m02 = ((v ** 2) * col_totals).sum() / N

m11 = 0.0
for i, ui in enumerate(u):
    for j, vj in enumerate(v):
        m11 += var_table[i, j] * ui * vj
m11 /= N

r_conditional = (m11 - m10 * m01) / math.sqrt((m20 - m10**2) * (m02 - m01**2))

df_r = pd.DataFrame([{
    "Метод вычисления": "Стандартная формула",
    "Выборочный коэффициент корреляции ($r_{xy}$)": round(r_standard, 5)
}, {
    "Метод вычисления": "Условные варианты",
    "Выборочный коэффициент корреляции ($r_{xy}$)": round(r_conditional, 5)
}])

display_markdown(df_r.to_markdown(index=False), raw=True)

| Метод вычисления    |   Выборочный коэффициент корреляции ($r_{xy}$) |
|:--------------------|-----------------------------------------------:|
| Стандартная формула |                                       -0.17818 |
| Условные варианты   |                                       -0.17818 |

Значение выборочного коэффициента корреляции, вычисленного по корреляционной таблице двумя способами **полностью совпадает**. Значение выборочного коэффициента корреляции свидетельствует о наличии слабой отрицательной корреляционной связи между признаками $X$ и $Y$.  

- Построить доверительный интервал для коэффициента корреляции при уровне значимости $ \gamma \in \{0.95, 0.99\} $, сделать выводы.


In [12]:
def calculate_correlation_ci(r: float, n: int, gamma: float) -> tuple[float, float]:
    """Расчёт доверительного интервала через преобразование Фишера"""
    z = 0.5 * np.log((1 + r) / (1 - r))
    se_z = 1 / np.sqrt(n - 3)
    
    alpha = 1 - gamma
    # Квантиль нормального распределения
    z_crit = stats.norm.ppf(1 - alpha / 2)
    
    ci_z_lower = z - z_crit * se_z
    ci_z_upper = z + z_crit * se_z
    
    ci_r_lower = (np.exp(2 * ci_z_lower) - 1) / (np.exp(2 * ci_z_lower) + 1)
    ci_r_upper = (np.exp(2 * ci_z_upper) - 1) / (np.exp(2 * ci_z_upper) + 1)
    
    return ci_r_lower, ci_r_upper

r = r_standard
ci_95 = calculate_correlation_ci(r, n, 0.95)
ci_99 = calculate_correlation_ci(r, n, 0.99)

df_ci = pd.DataFrame([{
    r"$\gamma$": 0.95,
    "Доверительный интервал для $r_{xy}$": f"({ci_95[0]:.4f}; {ci_95[1]:.4f})"
}, {
    r"$\gamma$": 0.99,
    "Доверительный интервал для $r_{xy}$": f"({ci_99[0]:.4f}; {ci_99[1]:.4f})"
}])

display_markdown(df_ci.to_markdown(index=False), raw=True)

|   $\gamma$ | Доверительный интервал для $r_{xy}$   |
|-----------:|:--------------------------------------|
|       0.95 | (-0.3560; 0.0121)                     |
|       0.99 | (-0.4076; 0.0724)                     |

Как и предполагается теоретически, с увеличением уровня доверительной вероятности $\gamma$ (от $0.95$ до $0.99$) ширина доверительного интервала возрастает для обеспечения более надежного накрытия неизвестного истинного значения коэффициента корреляции генеральной совокупности.

- Осуществить проверку статистической гипотезы о равенстве коэффициента корреляции нулю при заданном уровне значимости $ \alpha = 0.05 $, сделать выводы.

In [ ]:
def test_correlation_zero(r: float, n: int, alpha: float) -> dict[str, float | bool]:
    # t-статистика: распределение Стьюдента, df = n - 2
    t_stat = r * np.sqrt((n - 2) / (1 - r**2))
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 2)
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    
    t_stat = r * np.sqrt((n - 2) / (1 - r**2))
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 2)
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    
    return {
        't_statistic': t_stat,
        't_critical': t_crit,
        'p_value': p_value,
    }

alpha_val = 0.05
test_result = test_correlation_zero(r_standard, n, alpha_val)

display_markdown("### Проверка гипотезы $H_0: r = 0$ (отсутствие линейной корреляции)", raw=True)
display_markdown(f"- **Наблюдаемое значение ($t_{{набл}}$):** `{test_result['t_statistic']:.4f}`  \n" +
                 f"- **Критическое значение ($t_{{крит}}$):** `{test_result['t_critical']:.4f}`  \n" +
                 f"- **P-значение:** `{test_result['p_value']:.4e}`", raw=True)

### Проверка гипотезы $H_0: r = 0$ (отсутствие линейной корреляции)

- **Наблюдаемое значение ($t_{набл}$):** `-1.8555`  
- **Критическое значение ($t_{крит}$):** `1.9828`  
- **P-значение:** `6.6333e-02`

Так как $|t_{набл}| \le t_{крит}$, **нет оснований отвергать** нулевую гипотезу. Линейная корреляционная связь статистически незначима.

### Выводы
В результате выполнения лабораторной работы были освоены элементы корреляционного анализа и проверки соответствующих статистических гипотез:

1. Были обработаны выборочные данные по второму признаку (Ширина чашелистика), вычислены основные числовые характеристики (математическое ожидание, дисперсия, стандарное отклонение, асимметрия, эксцесс и коэффициент вариации).
2. Построен двумерный интервальный вариационный ряд для признаков $X$ (Длина чашелистика) и $Y$ (Ширина чашелистика), на основе которого выделена формализованная корреляционная таблица.
3. Значение выборочного коэффициента корреляции было вычислено двумя способами. Результаты совпали.
4. Построен доверительный интервал для найденного коэффициента корреляции с использованием преобразования Фишера. Рассмотрена динамика ширины интервала при изменениях доверительной вероятности $\gamma$.
5. Статистически проверена гипотеза $H_0: r = 0$ (об отсутствии корреляционной связи) по распределению Стьюдента. Наблюдаемое значение не превысило критическое, что даёт право подтвердить нулевую гипотезу с заданным уровнем ошибки $\alpha=0.05$.